Author: Dr. Víctor Uc Cetina

In [1]:
# Fasttext installation
!python3 --version
%pwd

# In a notebook, use %cd (plain "cd" is invalid Python).
# Do not use sudo; that is what asked for a password.
# The repo is already cloned under LLMs/Notebooks/fastText
%cd /Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks/fastText
!python3 -m pip install .
%cd /Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks

%pwd


Python 3.9.6
/Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks/fastText


/Users/victoruccetina/Library/Python/3.9/lib/python/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Defaulting to user installation because normal site-packages is not writeable
Processing ./.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for fasttext: filename=fasttext-0.9.2-cp39-cp39-macosx_26_0_universal2.whl size=673448 sha256=10aaec8752108e885c232c22461c950278914fee8c13f13670b93945bfb57dba
  Stored in directory: /private/var/folders/q8/1mz_tyzs0l97fh18rhtfz2v40000gn/T/pip-ephem-wheel-cache-qthcf7qn/wheels/a4/7d/1a/690117b60f178a41382bc807eda0677daffa27b3055e49e965
Successfully built fasttext
  Attempting uninstall: fasttext
    Found existing installation: fasttext 0.9.2
    Uninstalling fasttext-0.9.2:
      Successfully uninstalled fasttext-0.9.2
/Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks


'/Users/victoruccetina/Documents/code/inteligencia-artificial/LLMs/Notebooks'

In [2]:
# this cell takes 3 min to execute
import fasttext 
import fasttext.util

#from google.colab import drive
#drive.mount('/content/drive')

model_file = 'cc.es.300.bin'
ft = fasttext.load_model(model_file)
ft.get_dimension()

300

In [3]:
import math
from scipy.spatial import distance as cos_distance

def sorting_key(e):
  return e['distance']

def distance(vec1, vec2):
    dist = 0
    for i in range(len(vec1)):
        dist += (vec1[i]-vec2[i]) ** 2
    dist = math.sqrt( dist )
    return dist

def search(k, v1, entries, entries_vec, dist_metric):
    ## it calculates the k closest vectors to v1
    dist_list = list()
    idx_success = list()
    for i in range(len(entries)):
        e2 = entries[i]
        v2 = entries_vec[i]
        if dist_metric == "Euclidean":
          tmpDist = distance(v1,v2)
        elif dist_metric == "Cosine":
          tmpDist = cos_distance.cosine(v1,v2)
        dist_list.append({'idx': i, 'entry': e2, 'distance': tmpDist})
    dist_list.sort(key=sorting_key)
    for i in range(k):
        idx_success.append({'idx': dist_list[i].get('idx'), 'distance': dist_list[i].get('distance')})
    return idx_success

## We read the list of entries (sentences) from **KB.txt** and then we generate their corresponding embedding vectors of size 300.

In [4]:
entries = list()
f = open("KB-01.txt", "r")
for one_line in f:
  #print(one_line.strip())
  entries.append(one_line.strip())

entries_vec = list()
for e in entries:
  entries_vec.append(ft.get_sentence_vector(e))
  #entries_vec_gpt.append( funcion_gpt(e) )
  #entries_vec_bert.append( funcio_bert(e) )


## We define a query sentence, generate its embedding vector and search for the k most similar sentences. We can use two distance metrics: Euclidean or Cosine.

In [5]:
query = "Cuántas capas ocultas tiene una red neuronal?"
k = 10
distance_metric = "Cosine"
#distance_metric = "Euclidean"

query_vec = ft.get_sentence_vector(query)
#query_vec_gpt =
#query_vec_bert = 

idx_success = search(k, query_vec, entries, entries_vec, distance_metric)

top_results = list()
for i in range(len(idx_success)):
  top_results.append( [ idx_success[i]['idx'], entries[ idx_success[i]['idx'] ], idx_success[i]['distance'] ] )

for i in range(len(top_results)):
  print(top_results[i])

[552, 'Estos tipos de redes pueden implementarse con una sola capa de neuronas', np.float32(0.24268532)]
[288, 'Las neuronas de las capas ocultas pueden estar interconectadas de distintas maneras, lo que determina, junto con su número, las distintas topologías de redes neuronales', np.float32(0.2529168)]
[206, 'Una red individual puede ser entrenada para desarrollar una única y bien definida tarea (tareas complejas, que hagan múltiples selecciones de patrones, requerirán sistemas de redes interconectadas)', np.float32(0.25782698)]
[155, '3.2 Ventajas que ofrecen las red neuronal', np.float32(0.26008517)]
[230, 'Antes de comenzar el estudio sobre las redes neuronales, se debe aprender algo sobre las neuronas y de cómo ellas son utilizadas por una red neuronal', np.float32(0.27764904)]
[460, 'Estas dos posibilidades permiten distinguir entre dos tipos de redes con múltiples capas: las redes con conexiones hacia adelante o redes feedforward, y las redes que disponen de conexiones tanto ha

In [6]:
context = ""
for idx in range(len(top_results)):
  context += top_results[idx][1] + " "
print("Context:\n", context)

Context:
 Estos tipos de redes pueden implementarse con una sola capa de neuronas Las neuronas de las capas ocultas pueden estar interconectadas de distintas maneras, lo que determina, junto con su número, las distintas topologías de redes neuronales Una red individual puede ser entrenada para desarrollar una única y bien definida tarea (tareas complejas, que hagan múltiples selecciones de patrones, requerirán sistemas de redes interconectadas) 3.2 Ventajas que ofrecen las red neuronal Antes de comenzar el estudio sobre las redes neuronales, se debe aprender algo sobre las neuronas y de cómo ellas son utilizadas por una red neuronal Estas dos posibilidades permiten distinguir entre dos tipos de redes con múltiples capas: las redes con conexiones hacia adelante o redes feedforward, y las redes que disponen de conexiones tanto hacia adelante como hacia atrás o redes feedforward/feedback En las redes monocapa, se establecen conexiones entre las neuronas que pertenecen a la única capa que 